In [1]:
// load the formatting setup and the style sheet
#load "../setup.fsx"
HTML(System.IO.File.ReadAllText "../style.css")

The below script needs to be able to find the current output cell; this is an easy method to get it.

# Die Onkologie-Station

Auf einer Onkologie-Station liegen fünf Patienten in nebeneinander liegenden Zimmern.
Bis auf einen  der Patienten hat jeder genau eine Zigaretten-Marke geraucht.
Der Patient, der nicht Zigarette geraucht hat, hat Pfeife geraucht.
Jeder Patient fährt genau ein Auto und ist
an genau einer Krebs-Art erkrankt.  Zusätzlich haben Sie die folgenden Informationen:
<ol>
<li> Im Zimmer neben Michael wird Camel geraucht. </li>
<li> Der Trabant-Fahrer raucht Ernte 23 und liegt im Zimmer neben dem 
      Zungen-Krebs Patienten. </li>
<li> Rolf liegt im letzten Zimmer und hat Kehlkopf-Krebs. </li>
<li> Der West-Raucher liegt im ersten Zimmer. </li>
<li> Der Mazda-Fahrer hat Zungen-Krebs und liegt neben dem Trabant-Fahrer. </li>
<li> Der Nissan-Fahrer liegt neben dem Zungen-Krebs Patient. </li>
<li> Rudolf wünscht sich Sterbe-Hilfe und liegt zwischen dem Camel-Raucher und dem Trabant-Fahrer. </li>
<li> Der Seat Fahrer hat morgen seinen letzten Geburtstag. </li>
<li> Der Luckies Raucher liegt neben dem Patienten mit Lungen-Krebs. </li>
<li> Der Camel Raucher liegt neben dem Patienten mit Darm-Krebs. </li>
<li> Der Nissan Fahrer liegt neben dem Mazda-Fahrer. </li>
<li> Der Mercedes-Fahrer raucht Pfeife und liegt neben dem Camel Raucher. </li>
<li> Jens liegt neben dem Luckies Raucher. </li>
<li> Der Hodenkrebs-Patient hat gestern seine Eier durchs Klo gespült. </li>
</ol>
Entwickeln Sie ein <em>F#</em>-Programm, das die folgenden Fragen beantwortet:
<ol> 
<li> Was raucht der Darmkrebs-Patient? </li>
<li> Was fährt Kurt für ein Auto? </li>
</ol>

## Importing the Necessary Notebooks

Our goal is to solve this puzzle by first coding it as a solvability problem of propositional logic and then to solve the resulting set of clauses using the algorithm of Davis and Putnam.  We import the notebook `07-Davis-Putnam-JW.ipynb`.  This notebook imports the notebook `04-CNF.ipynb`, which implements the function `normalize` that takes a formula and transforms it into a set of clauses.  The notebook `04-CNF.ipynb` in turn imports the parser for propositional logic.

In [2]:
importNotebook "07-Davis-Putnam-JW.ipynb"

imported 

Propositional-Logic-Parser.ipynb, 04-CNF.ipynb, 07-Davis-Putnam-JW.ipynb

Using the parser and the function `normalize` we can implement the function `parseCNF`.  It takes one argument:
- `s` is a string representing a propositional formula.

It returns a set of clauses that is equivalent to the formula represented by `s`.

In [3]:
let parseCNF (s: string) : CNF =
    normalize (parse s)

## Auxiliary Functions

The function `atMostOne` takes one argument:
- `V` is a set of propositional variables.

It returns a set of clauses that is true if and only if at most one of the variables from $V$ is true.

In [4]:
let atMostOne (V: Set<string>) : CNF =
    set [ for p in V do
            for q in V do
                if p <> q then
                    set [ Neg p; Neg q ] ]

The function `var` takes two arguments:
- `f` is the name of a property, e.g. `"Japanese"`,
- `i` is the number of a room, i.e. $i \in\{1,2,3,4,5\}$.

It returns the string $f\langle i \rangle$, e.g. the call `var "Japanese" 2` returns the string `Japanese<2>`.

In [5]:
let var (f: string) (i: int) : string =
    sprintf "%s<%d>" f i

In [6]:
var "Japanese" 2

"Japanese<2>"

The function `flatten` takes one argument:
- `ListOfSets` is a list of sets.

It returns the union of these sets.

In [7]:
let flatten (ListOfSets: Set<'T> list) : Set<'T> =
    Set.unionMany ListOfSets

In [8]:
flatten [ set [1; 2; 3]; set [3; 4; 5] ]

set [1; 2; 3; 4; 5]

The function `somewhere` takes one argument:
- `x` is the name of a property.

It returns a clause that specifies that the person with property $x$ has to stay in one of the rooms from $1$ to $5$.

In [9]:
let somewhere (x: string) : Clause =
    set [ for i in 1 .. 5 -> Pos (var x i) ]

In [10]:
somewhere "a"

set [Pos "a<1>"; Pos "a<2>"; Pos "a<3>"; Pos "a<4>"; Pos "a<5>"]

The function `atMostOneAt` takes two arguments:
- `S` is a set of mutually exclusive properties,
- `i` is the number of a room.

It returns a set of clauses that specifies that the person staying in room number $i$ has at most one of the properties from the set $S$.  For example, if 
$S = \{\texttt{"Camel"}, \texttt{"Ernte"}, \texttt{"West"}, \texttt{"Luckies"}, \texttt{"Pfeife"}\}$, 
then $\texttt{atMostOneAt}(S, 3)$ specifies that the patient in room number 3 smokes at most one of these brands.

In [11]:
let atMostOneAt (S: Set<string>) (i: int) : CNF =
    atMostOne (Set.map (fun x -> var x i) S)

In [12]:
atMostOneAt (set [ "A"; "B"; "C" ]) 1

set [set [Neg "A<1>"; Neg "B<1>"]; set [Neg "A<1>"; Neg "C<1>"]; set [Neg "B<1>"; Neg "C<1>"]]

The function `onePerRoom` takes one argument:
- `S` is a set of five mutually exclusive properties.

It returns a set of clauses that expresses that for every property from $S$ there is a room where the patient with this property stays.  Furthermore, the set of clauses contains clauses that express that no patient has two of the properties from $S$.

In [13]:
let onePerRoom (S: Set<string>) : CNF =
    set [ for x in S -> somewhere x ] + flatten [ for i in 1 .. 5 -> atMostOneAt S i ]

In [14]:
onePerRoom (set [ "A"; "B"; "C"; "D"; "E" ])

set [set [Pos "A<1>"; Pos "A<2>"; Pos "A<3>"; Pos "A<4>"; Pos "A<5>"]; set [Pos "B<1>"; Pos "B<2>"; Pos "B<3>"; Pos "B<4>"; Pos "B<5>"]; set [Pos "C<1>"; Pos "C<2>"; Pos "C<3>"; Pos "C<4>"; Pos "C<5>"]; set [Pos "D<1>"; Pos "D<2>"; Pos "D<3>"; Pos "D<4>"; Pos "D<5>"]; set [Pos "E<1>"; Pos "E<2>"; Pos "E<3>"; Pos "E<4>"; Pos "E<5>"]; set [Neg "A<1>"; Neg "B<1>"]; set [Neg "A<1>"; Neg "C<1>"]; set [Neg "A<1>"; Neg "D<1>"]; set [Neg "A<1>"; Neg "E<1>"]; set [Neg "A<2>"; Neg "B<2>"]; set [Neg "A<2>"; Neg "C<2>"]; set [Neg "A<2>"; Neg "D<2>"]; set [Neg "A<2>"; Neg "E<2>"]; set [Neg "A<3>"; Neg "B<3>"]; set [Neg "A<3>"; Neg "C<3>"]; set [Neg "A<3>"; Neg "D<3>"]; set [Neg "A<3>"; Neg "E<3>"]; set [Neg "A<4>"; Neg "B<4>"]; set [Neg "A<4>"; Neg "C<4>"]; set [Neg "A<4>"; Neg "D<4>"]; set [Neg "A<4>"; Neg "E<4>"]; set [Neg "A<5>"; Neg "B<5>"]; set [Neg "A<5>"; Neg "C<5>"]; set [Neg "A<5>"; Neg "D<5>"]; set [Neg "A<5>"; Neg "E<5>"]; set [Neg "B<1>"; Neg "C<1>"]; set [Neg "B<1>"; Neg "D<1>"]; set [

The function `sameRoom` takes two arguments:
- `a` is a property,
- `b` is a property.

It returns a set of clauses that specifies that if the patient in room number $i$ has the property $a$, then he also has the property $b$ and vice versa.

In [15]:
let sameRoom (a: string) (b: string) : CNF =
    flatten [ for i in 1 .. 5 -> parseCNF (sprintf "%s ↔ %s" (var a i) (var b i)) ]

In [16]:
sameRoom "Luckies" "Darm"

set [set [Pos "Darm<1>"; Neg "Luckies<1>"]; set [Pos "Darm<2>"; Neg "Luckies<2>"]; set [Pos "Darm<3>"; Neg "Luckies<3>"]; set [Pos "Darm<4>"; Neg "Luckies<4>"]; set [Pos "Darm<5>"; Neg "Luckies<5>"]; set [Pos "Luckies<1>"; Neg "Darm<1>"]; set [Pos "Luckies<2>"; Neg "Darm<2>"]; set [Pos "Luckies<3>"; Neg "Darm<3>"]; set [Pos "Luckies<4>"; Neg "Darm<4>"]; set [Pos "Luckies<5>"; Neg "Darm<5>"]]

The function `differentRoom` takes two arguments:
- `a` is a property,
- `b` is a property.

It returns a set of clauses that specifies that the patients with properties $a$ and $b$ stay in different rooms. 

In [17]:
let differentRoom (a: string) (b: string) : CNF =
    flatten [ for i in 1 .. 5 -> parseCNF (sprintf "%s → ¬%s" (var a i) (var b i)) ]

In [18]:
differentRoom "A" "B"

set [set [Neg "A<1>"; Neg "B<1>"]; set [Neg "A<2>"; Neg "B<2>"]; set [Neg "A<3>"; Neg "B<3>"]; set [Neg "A<4>"; Neg "B<4>"]; set [Neg "A<5>"; Neg "B<5>"]]

The function `nextTo` takes two arguments:
- `a` is a property,
- `b` is a property.

It returns a set of clauses that specifies that the patients with properties $a$ and $b$ stay in neighbouring rooms.

In [19]:
let nextTo (a: string) (b: string) : CNF =
    parseCNF (sprintf "%s → %s" (var a 1) (var b 2))
    + flatten [ for i in 2 .. 4 ->
                  parseCNF (sprintf "%s → %s ∨ %s" (var a i) (var b (i-1)) (var b (i+1))) ]
    + parseCNF (sprintf "%s → %s" (var a 5) (var b 4))

In [20]:
nextTo "A" "B"

set [set [Pos "B<1>"; Pos "B<3>"; Neg "A<2>"]; set [Pos "B<2>"; Pos "B<4>"; Neg "A<3>"]; set [Pos "B<2>"; Neg "A<1>"]; set [Pos "B<3>"; Pos "B<5>"; Neg "A<4>"]; set [Pos "B<4>"; Neg "A<5>"]]

In [21]:
let Brands  = set [ "Camel"; "Ernte"; "West"; "Luckies"; "Pfeife" ]
let Cars    = set [ "Trabant"; "Mazda"; "Nissan"; "Seat"; "Mercedes" ]
let Cancers = set [ "Zunge"; "Kehlkopf"; "Lunge"; "Darm"; "Hoden" ]
let Names   = set [ "Michael"; "Rolf"; "Rudolf"; "Jens"; "Kurt" ]

The function `allClauses` takes no arguments.  It returns a set of clauses describing the problem.

In [22]:
let allClauses () : CNF =
    let mutable Clauses = onePerRoom Brands
    Clauses <- Clauses + onePerRoom Cars
    Clauses <- Clauses + onePerRoom Cancers
    Clauses <- Clauses + onePerRoom Names
    // Im Zimmer neben Michael wird Camel geraucht. 
    Clauses <- Clauses + nextTo "Michael" "Camel"
    // Der Trabant-Fahrer raucht Ernte 23 und liegt im Zimmer neben dem 
    // Zungen-Krebs Patienten. 
    Clauses <- Clauses + sameRoom "Trabant" "Ernte"
    Clauses <- Clauses + nextTo "Trabant" "Zunge"
    // Rolf liegt im letzten Zimmer und hat Kehlkopf-Krebs.
    Clauses <- Clauses + sameRoom "Rolf" "Kehlkopf"
    Clauses <- Clauses + parseCNF "Rolf<5>"
    // Der West-Raucher liegt im ersten Zimmer. 
    Clauses <- Clauses + parseCNF "West<1>"
    // Der Mazda-Fahrer hat Zungen-Krebs und liegt neben dem Trabant-Fahrer. 
    Clauses <- Clauses + sameRoom "Mazda" "Zunge"
    Clauses <- Clauses + nextTo "Mazda" "Trabant"
    // Der Nissan-Fahrer liegt neben dem Zungen-Krebs Patient. 
    Clauses <- Clauses + nextTo "Nissan" "Zunge"
    // Rudolf wünscht sich Sterbe-Hilfe und liegt zwischen dem Camel-Raucher und dem Trabant-Fahrer. 
    Clauses <- Clauses + nextTo "Rudolf" "Camel"
    Clauses <- Clauses + nextTo "Rudolf" "Trabant"
    Clauses <- Clauses + differentRoom "Camel" "Trabant"
    // Der Luckies Raucher liegt neben dem Patienten mit Lungen-Krebs. 
    Clauses <- Clauses + nextTo "Luckies" "Lunge"
    // Der Camel Raucher liegt neben dem Patienten mit Darm-Krebs. 
    Clauses <- Clauses + nextTo "Camel" "Darm"
    // Der Nissan Fahrer liegt neben dem Mazda-Fahrer. 
    Clauses <- Clauses + nextTo "Nissan" "Mazda"
    // Der Mercedes-Fahrer raucht Pfeife und liegt neben dem Camel Raucher. 
    Clauses <- Clauses + sameRoom "Mercedes" "Pfeife"
    Clauses <- Clauses + nextTo "Mercedes" "Camel"
    // Jens liegt neben dem Luckies Raucher. 
    Clauses <- Clauses + nextTo "Jens" "Luckies"
    // Der Hodenkrebs-Patient hat gestern seine Eier durchs Klo gespült.
    Clauses

In [23]:
let Clauses = allClauses ()
printfn "%s" (cnfToString Clauses)

{{Camel<1>, Camel<2>, Camel<3>, Camel<4>, Camel<5>}, {Camel<1>, Camel<3>, ¬Mercedes<2>}, {Camel<1>, Camel<3>, ¬Michael<2>}, {Camel<1>, Camel<3>, ¬Rudolf<2>}, {Camel<2>, Camel<4>, ¬Mercedes<3>}, {Camel<2>, Camel<4>, ¬Michael<3>}, {Camel<2>, Camel<4>, ¬Rudolf<3>}, {Camel<2>, ¬Mercedes<1>}, {Camel<2>, ¬Michael<1>}, {Camel<2>, ¬Rudolf<1>}, {Camel<3>, Camel<5>, ¬Mercedes<4>}, {Camel<3>, Camel<5>, ¬Michael<4>}, {Camel<3>, Camel<5>, ¬Rudolf<4>}, {Camel<4>, ¬Mercedes<5>}, {Camel<4>, ¬Michael<5>}, {Camel<4>, ¬Rudolf<5>}, {Darm<1>, Darm<2>, Darm<3>, Darm<4>, Darm<5>}, {Darm<1>, Darm<3>, ¬Camel<2>}, {Darm<2>, Darm<4>, ¬Camel<3>}, {Darm<2>, ¬Camel<1>}, {Darm<3>, Darm<5>, ¬Camel<4>}, {Darm<4>, ¬Camel<5>}, {Ernte<1>, Ernte<2>, Ernte<3>, Ernte<4>, Ernte<5>}, {Ernte<1>, ¬Trabant<1>}, {Ernte<2>, ¬Trabant<2>}, {Ernte<3>, ¬Trabant<3>}, {Ernte<4>, ¬Trabant<4>}, {Ernte<5>, ¬Trabant<5>}, {Hoden<1>, Hoden<2>, Hoden<3>, Hoden<4>, Hoden<5>}, {Jens<1>, Jens<2>, Jens<3>, Jens<4>, Jens<5>}, {Kehlkopf<1>, Kehlkopf

In [24]:
Clauses.Count

322

Solving the problem takes less than a second.

In [25]:
#!time
let Solution = solve Clauses

Wall time: 26.4411ms

## Pretty Printing the Solution

The function `extractAssignment` takes one argument:
- `Solution` is a set of unit clauses representing a solution of the puzzle.

It returns a map that assigns to every property the number of the room of the patient having this property.  Only the positive literals are relevant.  The function `arb` is defined in the notebook `07-Davis-Putnam-JW.ipynb`.

In [26]:
let extractAssignment (Solution: CNF) : Map<string, int> =
    Map [ for Unit in Solution do
            match arb Unit with
            | Pos v -> let k = v.IndexOf '<'
                       (v.Substring(0, k), int (v.Substring(k + 1, v.Length - k - 2)))
            | Neg _ -> () ]

The function `showHTML` takes one argument:
- `Assignment` is a map assigning room numbers to properties.

It returns an HTML table that shows the solution.  The table has one row for every room.

In [27]:
let showHTML (Assignment: Map<string, int>) =
    let sb = System.Text.StringBuilder()
    sb.Append("<table style=\"border:2px solid blue\">\n<tr>") |> ignore
    for name in [ "Room"; "Brands"; "Cars"; "Cancers"; "Names" ] do
        sb.Append("<th style=\"color:gold; background-color:blue\">" + name + "</th>") |> ignore
    sb.Append("</tr>\n") |> ignore
    for room in 1 .. 5 do
        sb.Append("<tr><td style=\"border:1px solid green\">" + string room + "</td>") |> ignore
        for Class in [ Brands; Cars; Cancers; Names ] do
            for x in Class do
                if Assignment[x] = room then
                    sb.Append("<td style=\"border:1px solid green\">" + x + "</td>") |> ignore
        sb.Append("</tr>\n") |> ignore
    sb.Append("</table>") |> ignore
    HTML(sb.ToString())

In [28]:
showHTML (extractAssignment Solution)

Room,Brands,Cars,Cancers,Names
1,West,Seat,Hoden,Kurt
2,Pfeife,Mercedes,Darm,Michael
3,Camel,Nissan,Lunge,Jens
4,Luckies,Mazda,Zunge,Rudolf
5,Ernte,Trabant,Kehlkopf,Rolf


## Checking the Uniqueness of the Solution

The function `negateSolution` takes one argument:
- `UnitClauses` is a set of unit clauses.

It returns a clause that is the negation of the conjunction of these unit clauses.

In [29]:
let negateSolution (UnitClauses: CNF) : Clause =
    Set.map (fun Unit -> complement (arb Unit)) UnitClauses

In [30]:
negateSolution (set [ set [ Pos "a" ]; set [ Neg "b" ] ])

set [Pos "b"; Neg "a"]

The function `checkUniqueness` takes two arguments:
- `Solution` is a set of unit clauses representing a solution of `Clauses`,
- `Clauses` is a set of clauses.

It checks whether `Solution` is the only solution of `Clauses`.  To this end, the negation of `Solution` is added to `Clauses`.  If the resulting set of clauses is unsatisfiable, the solution is unique.

In [31]:
let checkUniqueness (Solution: CNF) (Clauses: CNF) : unit =
    let negation    = negateSolution Solution
    let alternative = solve (Clauses.Add negation)
    if alternative = set [ Set.empty ] then
        printfn "Well done: The solution is unique!"
    else
        printfn "ERROR: The solution is not unique!"

In [32]:
checkUniqueness Solution Clauses

Well done: The solution is unique!